# Mục lục

## Cài đặt thư viện

In [8]:
!pip install git+https://github.com/thinh-vu/vnstock

  Cloning https://github.com/thinh-vu/vnstock to /tmp/pip-req-build-w4nur9c5
  Running command git clone --filter=blob:none --quiet https://github.com/thinh-vu/vnstock /tmp/pip-req-build-w4nur9c5
  Resolved https://github.com/thinh-vu/vnstock to commit 7bc8ec5eeda5c107995975d985306405ccbc512e
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## Nạp thư viện lần đầu

In [9]:
from vnstock import Listing, Company, Finance, Trading, Screener

# II. Truy xuất dữ liệu - CK Việt Nam

## Listing: Liệt kê danh sách chứng khoán

In [10]:
from vnstock import Listing
listing = Listing()

### Danh sách cổ phiếu theo nhóm

In [11]:
# Liệt kê tất cả mã chứng khoán theo nhóm phân loại. Ví dụ HOSE, VN30, VNMidCap, VNSmallCap, VNAllShare, VN100, ETF, HNX, HNX30, HNXCon, HNXFin, HNXLCap, HNXMSCap, HNXMan, UPCOM, FU_INDEX (mã chỉ số hợp đồng tương lai), CW (chứng quyền)
listing.symbols_by_group('VN30')

0     ACB
1     BCM
2     BID
3     CTG
4     DGC
5     FPT
6     GAS
7     GVR
8     HDB
9     HPG
10    LPB
11    MBB
12    MSN
13    MWG
14    PLX
15    SAB
16    SHB
17    SSB
18    SSI
19    STB
20    TCB
21    TPB
22    VCB
23    VHM
24    VIB
25    VIC
26    VJC
27    VNM
28    VPB
29    VRE
Name: symbol, dtype: object

## Quote: Giá chứng khoán

In [12]:
from vnstock import Quote
import os
quote = Quote(symbol='VN30', source='TCBS')

#### Giá lịch sử

##### Cổ phiếu

In [13]:
#ACB
from vnstock import Quote
quote = Quote(symbol='ACB', source='TCBS')

quote.history(symbol='', start='2020-01-01', end='2025-11-10', interval='1D')

,time,open,high,low,close,volume
3,2020-01-02,6.58,6.64,6.55,6.64,1163109
4,2020-01-03,6.64,6.70,6.61,6.64,1055528
5,2020-01-06,6.64,6.64,6.49,6.49,1286035
6,2020-01-07,6.49,6.55,6.49,6.49,1050934
7,2020-01-08,6.49,6.49,6.35,6.38,2304937
...,...,...,...,...,...,...
1495,2025-11-04,25.45,26.10,25.40,26.00,17988658
1496,2025-11-05,25.80,25.90,25.55,25.60,6234065
1497,2025-11-06,25.55,25.70,25.30,25.30,6532302
1498,2025-11-07,25.25,25.30,24.85,24.90,12120670


In [ ]:
# VN30 2025
from vnstock import Quote
import os, time, json, re, sys
from tqdm import tqdm
import pandas as pd

# Cấu hình
save_folder = os.path.join("docs", "Data_VN30")
os.makedirs(save_folder, exist_ok=True)

progress_file = os.path.join(save_folder, "progress.json")
start_date = '2020-01-01'
end_date   = '2025-11-10'
source     = 'TCBS'   # có thể đổi thành 'VCI' nếu muốn thử nguồn khác
interval   = '1D'

vn30_symbols = [
    'ACB', 'BCM', 'BID', 'CTG', 'DGC', 'FPT', 'GAS', 'GVR', 'HDB', 'HPG',
    'LPB', 'MBB', 'MSN', 'MWG', 'PLX', 'SAB', 'SHB', 'SSB', 'SSI', 'STB',
    'TCB', 'TPB', 'VCB', 'VHM', 'VIB', 'VIC', 'VJC', 'VNM', 'VPB', 'VRE']

# Helpers
def load_progress(fn):
    if os.path.exists(fn):
        try:
            return json.load(open(fn, 'r'))
        except:
            return {}
    return {}

def save_progress(fn, data):
    with open(fn, 'w') as f:
        json.dump(data, f, indent=2)

def parse_wait_seconds_from_msg(msg):
    if not msg:
        return None
    # tìm pattern tiếng việt "xx giây"
    m = re.search(r'(\d+)\s*giây', msg, flags=re.IGNORECASE)
    if m:
        return int(m.group(1))
    # tìm pattern tiếng anh "xx seconds" hoặc "retry after xx"
    m2 = re.search(r'(?:(\d+)\s*seconds)|(?:retry after[: ]?(\d+))', msg, flags=re.IGNORECASE)
    if m2:
        for g in m2.groups():
            if g and g.isdigit():
                return int(g)
    return None

# Hàm tải với bắt SystemExit
def fetch_symbol_to_file(symbol,
                         start=start_date,
                         end=end_date,
                         source=source,
                         interval=interval,
                         out_folder=save_folder,
                         max_attempts=8,
                         initial_backoff=2.0):
    out_path = os.path.join(out_folder, f"{symbol}_{start.replace('-','')}_{end.replace('-','')}.xlsx")
    if os.path.exists(out_path):
        return True, out_path  # đã có, bỏ qua

    attempt = 0
    backoff = initial_backoff

    while attempt < max_attempts:
        attempt += 1
        try:
            # Thực hiện call
            quote = Quote(symbol=symbol, source=source)
            df = quote.history(symbol=symbol, start=start, end=end, interval=interval)

            # Nếu trả về None hoặc empty -> coi như valid nhưng báo warning
            if df is None or df.empty:
                print(f"⚠️ {symbol}: no data returned. Skipping save.")
                return True, None

            # Giữ chỉ các cột cần thiết nếu có
            keep_cols = [c for c in ['time','open','close','volume'] if c in df.columns]
            if keep_cols:
                df = df[keep_cols]

            df.to_excel(out_path, index=False)
            print(f"✅ Saved {symbol} -> {out_path}")
            return True, out_path

        except SystemExit as se:
            # Bắt SystemExit => thư viện muốn terminate; thay vào đó parse thông điệp và sleep
            msg = str(se)
            wait_sec = parse_wait_seconds_from_msg(msg)
            if wait_sec:
                print(f"⏳ SystemExit caught for {symbol}. Server asked to wait {wait_sec}s. Sleeping...")
                time.sleep(wait_sec + 1)
            else:
                # fallback backoff
                print(f"⏳ SystemExit caught for {symbol}. No wait parsed. Backoff {backoff:.1f}s (attempt {attempt}/{max_attempts})")
                time.sleep(backoff)
                backoff *= 2

        except Exception as e:
            # Bắt các Exception khác (requests, ValueError, RateLimit trong message)
            msg = str(e)
            wait_sec = parse_wait_seconds_from_msg(msg)
            if wait_sec:
                print(f"⏳ Rate-limit message for {symbol}: wait {wait_sec}s. Sleeping...")
                time.sleep(wait_sec + 1)
            elif '429' in msg or 'rate limit' in msg.lower() or 'too many requests' in msg.lower():
                print(f"⏳ Rate-limit detected for {symbol}. Backoff {backoff:.1f}s (attempt {attempt}/{max_attempts})")
                time.sleep(backoff)
                backoff *= 2
            else:
                print(f"⚠️ Error for {symbol} (attempt {attempt}/{max_attempts}): {msg}")
                # những lỗi không phải rate limit: vẫn thử vài lần
                time.sleep(backoff)
                backoff *= 2

    print(f"❌ Give up {symbol} after {max_attempts} attempts.")
    return False, None

# Orchestrator chính (resume-friendly)
progress = load_progress(progress_file)
if not progress:
    progress = {"done": [], "failed": []}

for sym in tqdm(vn30_symbols, desc="VN30 all"):
    if sym in progress.get("done", []):
        continue  # đã xong trước đó

    ok, path = fetch_symbol_to_file(sym)
    if ok:
        progress.setdefault("done", []).append(sym)
    else:
        progress.setdefault("failed", []).append(sym)

    save_progress(progress_file, progress)  # lưu progress liên tục

print("Finished. Done:", len(progress.get("done", [])), "Failed:", len(progress.get("failed", [])))
print("Files saved to:", save_folder)

In [3]:
# Các mã bị loại
from vnstock import Quote
import os, time, json, re, sys
from tqdm import tqdm
import pandas as pd

# Cấu hình
save_folder = os.path.join("docs", "Eliminated_VN30")
os.makedirs(save_folder, exist_ok=True)

progress_file = os.path.join(save_folder, "progress.json")
start_date = '2020-01-01'
end_date   = '2025-11-10'
source     = 'TCBS'   # có thể đổi thành 'VCI' nếu muốn thử nguồn khác
interval   = '1D'

vn30_symbols = [
    'BVH', 'EIB', 'KDH', 'NVL', 'PDR', 'PNJ', 'REE', 'ROS', 'SBT', 'TCH'
]

# Helpers
def load_progress(fn):
    if os.path.exists(fn):
        try:
            return json.load(open(fn, 'r'))
        except:
            return {}
    return {}

def save_progress(fn, data):
    with open(fn, 'w') as f:
        json.dump(data, f, indent=2)

def parse_wait_seconds_from_msg(msg):
    if not msg:
        return None
    # tìm pattern tiếng việt "xx giây"
    m = re.search(r'(\d+)\s*giây', msg, flags=re.IGNORECASE)
    if m:
        return int(m.group(1))
    # tìm pattern tiếng anh "xx seconds" hoặc "retry after xx"
    m2 = re.search(r'(?:(\d+)\s*seconds)|(?:retry after[: ]?(\d+))', msg, flags=re.IGNORECASE)
    if m2:
        for g in m2.groups():
            if g and g.isdigit():
                return int(g)
    return None

# Hàm tải với bắt SystemExit
def fetch_symbol_to_file(symbol,
                         start=start_date,
                         end=end_date,
                         source=source,
                         interval=interval,
                         out_folder=save_folder,
                         max_attempts=8,
                         initial_backoff=2.0):
    out_path = os.path.join(out_folder, f"{symbol}_{start.replace('-','')}_{end.replace('-','')}.xlsx")
    if os.path.exists(out_path):
        return True, out_path  # đã có, bỏ qua

    attempt = 0
    backoff = initial_backoff

    while attempt < max_attempts:
        attempt += 1
        try:
            # Thực hiện call
            quote = Quote(symbol=symbol, source=source)
            df = quote.history(symbol=symbol, start=start, end=end, interval=interval)

            # Nếu trả về None hoặc empty -> coi như valid nhưng báo warning
            if df is None or df.empty:
                print(f"⚠️ {symbol}: no data returned. Skipping save.")
                return True, None

            # Giữ chỉ các cột cần thiết nếu có
            keep_cols = [c for c in ['time','open','close','volume'] if c in df.columns]
            if keep_cols:
                df = df[keep_cols]

            df.to_excel(out_path, index=False)
            print(f"✅ Saved {symbol} -> {out_path}")
            return True, out_path

        except SystemExit as se:
            # Bắt SystemExit => thư viện muốn terminate; thay vào đó parse thông điệp và sleep
            msg = str(se)
            wait_sec = parse_wait_seconds_from_msg(msg)
            if wait_sec:
                print(f"⏳ SystemExit caught for {symbol}. Server asked to wait {wait_sec}s. Sleeping...")
                time.sleep(wait_sec + 1)
            else:
                # fallback backoff
                print(f"⏳ SystemExit caught for {symbol}. No wait parsed. Backoff {backoff:.1f}s (attempt {attempt}/{max_attempts})")
                time.sleep(backoff)
                backoff *= 2

        except Exception as e:
            # Bắt các Exception khác (requests, ValueError, RateLimit trong message)
            msg = str(e)
            wait_sec = parse_wait_seconds_from_msg(msg)
            if wait_sec:
                print(f"⏳ Rate-limit message for {symbol}: wait {wait_sec}s. Sleeping...")
                time.sleep(wait_sec + 1)
            elif '429' in msg or 'rate limit' in msg.lower() or 'too many requests' in msg.lower():
                print(f"⏳ Rate-limit detected for {symbol}. Backoff {backoff:.1f}s (attempt {attempt}/{max_attempts})")
                time.sleep(backoff)
                backoff *= 2
            else:
                print(f"⚠️ Error for {symbol} (attempt {attempt}/{max_attempts}): {msg}")
                # những lỗi không phải rate limit: vẫn thử vài lần
                time.sleep(backoff)
                backoff *= 2

    print(f"❌ Give up {symbol} after {max_attempts} attempts.")
    return False, None

# Orchestrator chính (resume-friendly)
progress = load_progress(progress_file)
if not progress:
    progress = {"done": [], "failed": []}

for sym in tqdm(vn30_symbols, desc="VN30 all"):
    if sym in progress.get("done", []):
        continue  # đã xong trước đó

    ok, path = fetch_symbol_to_file(sym)
    if ok:
        progress.setdefault("done", []).append(sym)
    else:
        progress.setdefault("failed", []).append(sym)

    save_progress(progress_file, progress)  # lưu progress liên tục

print("Finished. Done:", len(progress.get("done", [])), "Failed:", len(progress.get("failed", [])))
print("Files saved to:", save_folder)

VN30 all: 100%|██████████| 20/20 [00:00<00:00, 227333.55it/s]

Finished. Done: 20 Failed: 0
Files saved to: docs/Elimiated_VN30


In [5]:
# ZIP ĐỂ TẢI
!zip -r E_VN30_Data.zip docs/Eliminated_VN30

  adding: docs/Elimiated_VN30/ (stored 0%)
  adding: docs/Elimiated_VN30/TCH_20200101_20251110.xlsx (deflated 4%)
  adding: docs/Elimiated_VN30/PNJ_20200101_20251110.xlsx (deflated 2%)
  adding: docs/Elimiated_VN30/KDH_20200101_20251110.xlsx (deflated 5%)
  adding: docs/Elimiated_VN30/EIB_20200101_20251110.xlsx (deflated 4%)
  adding: docs/Elimiated_VN30/progress.json (deflated 54%)
  adding: docs/Elimiated_VN30/.ipynb_checkpoints/ (stored 0%)
  adding: docs/Elimiated_VN30/NVL_20200101_20251110.xlsx (deflated 3%)
  adding: docs/Elimiated_VN30/ROS_20200101_20251110.xlsx (deflated 4%)
  adding: docs/Elimiated_VN30/BVH_20200101_20251110.xlsx (deflated 4%)
  adding: docs/Elimiated_VN30/SBT_20200101_20251110.xlsx (deflated 5%)
  adding: docs/Elimiated_VN30/REE_20200101_20251110.xlsx (deflated 4%)
  adding: docs/Elimiated_VN30/PDR_20200101_20251110.xlsx (deflated 5%)


### VN30

In [10]:
from vnstock import Quote
quote = Quote(symbol='VN30', source='TCBS')

In [11]:
quote.history(symbol='', start='2020-01-01', end='2025-11-10', interval='1D')

,time,open,high,low,close,volume
0,2020-01-06,882.51,882.51,872.34,872.34,36868840
1,2020-01-07,872.32,877.34,870.49,876.70,40277740
2,2020-01-08,876.64,876.64,861.51,865.18,53101180
3,2020-01-09,877.47,878.43,865.18,877.76,45705720
4,2020-01-10,877.85,888.09,877.70,885.52,72856040
...,...,...,...,...,...,...
1495,2025-11-04,1860.56,1908.55,1843.43,1897.71,459231368
1496,2025-11-05,1888.34,1896.94,1878.75,1886.47,234523664
1497,2025-11-06,1886.48,1890.96,1866.91,1869.60,233036138
1498,2025-11-07,1862.19,1864.91,1817.93,1824.71,395069671


In [12]:
df = quote.history(symbol='', start='2020-01-01', end='2025-11-10', interval='1D')
df.to_excel('VN30_2020_2025.xlsx', index=False)
print("Đã lưu file thành công!")

Đã lưu file thành công!


## Merge datasets

In [10]:
import pandas as pd
import os

def merge_all_vn30_with_index(base_folder="/workspaces/vnstock/docs/Data", output_file="All_VN30_Merged.xlsx"):
    """
    Gộp toàn bộ dữ liệu:
      - /workspaces/vnstock/docs/Data/Data_VN30/
      - /workspaces/vnstock/docs/Data/Eliminated_VN30/
      - /workspaces/vnstock/docs/Data/VN30_2020_2025.xlsx
    Thành 1 file Excel duy nhất.
    """
    folders = ["Data_VN30", "Eliminated_VN30"]
    merged_data = []

    # 1️⃣ Đọc dữ liệu trong hai thư mục cổ phiếu
    for folder in folders:
        folder_path = os.path.join(base_folder, folder)
        if not os.path.exists(folder_path):
            print(f"⚠️ Thư mục không tồn tại: {folder_path}")
            continue

        excel_files = [f for f in os.listdir(folder_path) if f.endswith(('.xlsx', '.xls'))]
        print(f"📂 Đang đọc {len(excel_files)} file trong {folder_path}...")

        for file in excel_files:
            file_path = os.path.join(folder_path, file)
            try:
                df = pd.read_excel(file_path)
                # Lấy mã cổ phiếu từ tên file: ACB_20200101_20251110.xlsx → ACB
                symbol = os.path.splitext(file)[0].split("_")[0]
                df["symbol"] = symbol
                df["group"] = folder
                merged_data.append(df)
            except Exception as e:
                print(f"⚠️ Lỗi đọc file {file_path}: {e}")

    # 2️⃣ Đọc file chỉ số VN30 riêng
    index_file = os.path.join(base_folder, "VN30_2020_2025.xlsx")
    if os.path.exists(index_file):
        try:
            df_index = pd.read_excel(index_file)
            df_index["symbol"] = "VN30"
            df_index["group"] = "Index"
            merged_data.append(df_index)
            print(f"✅ Đã thêm file chỉ số VN30: {index_file}")
        except Exception as e:
            print(f"⚠️ Lỗi đọc file VN30 index: {e}")
    else:
        print(f"⚠️ Không tìm thấy file VN30 index tại: {index_file}")

    # 3️⃣ Gộp dữ liệu
    if not merged_data:
        print("❌ Không có dữ liệu nào để gộp.")
        return

    merged_df = pd.concat(merged_data, ignore_index=True)

    # 4️⃣ Chuẩn hóa cột thời gian và sắp xếp
    if "time" in merged_df.columns:
        merged_df["time"] = pd.to_datetime(merged_df["time"], errors="coerce")
        merged_df = merged_df.sort_values(["symbol", "time"]).reset_index(drop=True)

    # 5️⃣ Lưu ra file Excel
    output_path = os.path.join(base_folder, output_file)
    merged_df.to_excel(output_path, index=False)
    print(f"\n✅ Đã gộp thành công {len(merged_data)} tập dữ liệu.")
    print(f"📄 File đầu ra: {output_path}")
    print(f"📊 Tổng số dòng: {len(merged_df)}")

# 👉 Chạy hàm
merge_all_vn30_with_index()

📂 Đang đọc 38 file trong /workspaces/vnstock/docs/Data/Data_VN30...
📂 Đang đọc 10 file trong /workspaces/vnstock/docs/Data/Eliminated_VN30...
✅ Đã thêm file chỉ số VN30: /workspaces/vnstock/docs/Data/VN30_2020_2025.xlsx

✅ Đã gộp thành công 49 tập dữ liệu.
📄 File đầu ra: /workspaces/vnstock/docs/Data/All_VN30_Merged.xlsx
📊 Tổng số dòng: 73153
